# VAR Model Tutorial

This notebook demonstrates how to fit a **Vector Autoregression (VAR)** model
using the custom numpy-based `VARModel` class.

## What is VAR?

A VAR(p) model generalises the univariate AR model to *multiple* time series.
Each variable is modelled as a linear function of its own past values **and**
the past values of all other variables in the system:

$$\mathbf{Y}_t = \mathbf{c} + \sum_{i=1}^{p} \mathbf{A}_i \mathbf{Y}_{t-i} + \mathbf{u}_t$$

where $\mathbf{Y}_t$ is a vector of $K$ variables and $\mathbf{A}_i$ are
$K \times K$ coefficient matrices.

## Dataset

We use Indian stock and index monthly data from the repository's `sample_data` module.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'testing'))

import numpy as np
import matplotlib.pyplot as plt

from timeseries import moving_average, difference, autocorrelation, VARModel
from sample_data import (RELIANCE_MONTHLY, TCS_MONTHLY, NIFTY50_MONTHLY,
                         MONTHS_2023)

## 1. Visualise the data

In [ ]:
columns = ['Reliance', 'TCS', 'Nifty50']
raw = [RELIANCE_MONTHLY, TCS_MONTHLY, NIFTY50_MONTHLY]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, name, series in zip(axes, columns, raw):
    ax.plot(MONTHS_2023, series, marker='o')
    ax.set_title(name)
plt.suptitle('Indian Stocks / Index — Monthly 2023', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 2. Differencing for stationarity

In [ ]:
# Log-difference each series
diff_data = {}
for name, series in zip(columns, raw):
    log_vals = np.log(series).tolist()
    diff_data[name] = difference(log_vals, lag=1)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, name in zip(axes, columns):
    ax.plot(MONTHS_2023[1:], diff_data[name], marker='o')
    ax.set_title(f'Δlog({name})')
    ax.axhline(0, color='grey', linestyle='--')
plt.suptitle('Log-Differenced Series', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation of each differenced series
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, columns):
    vals = diff_data[name]
    max_lag = min(5, len(vals) - 1)
    acf_vals = [autocorrelation(vals, lag) for lag in range(1, max_lag + 1)]
    ax.bar(range(1, max_lag + 1), acf_vals)
    ax.set_title(f'ACF of Δlog({name})')
    ax.set_xlabel('Lag')
plt.tight_layout()
plt.show()

## 3. Fit the VAR model (numpy-based)

In [ ]:
# Stack into a (T, 3) array
Y = np.column_stack([diff_data[c] for c in columns])
print(f'Multivariate series shape: {Y.shape}')

model = VARModel(p=1)
model.fit(Y)

print(f'\nIntercept: {np.round(model.intercept, 6)}')
print(f'\nCoefficient matrix A1 (lag 1):')
print(np.round(model.coefficients[0], 6))

## 4. Forecasting

In [ ]:
n_forecast = 3
fc = model.forecast(steps=n_forecast)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, columns)):
    obs = diff_data[name]
    ax.plot(range(len(obs)), obs, marker='o', label='Observed')
    ax.plot(range(len(obs), len(obs) + n_forecast), fc[:, i],
            marker='s', color='red', label='Forecast')
    ax.set_title(f'Δlog({name})')
    ax.legend(fontsize=8)
plt.suptitle('VAR(1) Forecast', fontsize=14)
plt.tight_layout()
plt.show()

print('Forecasted values:')
for i, name in enumerate(columns):
    print(f'  {name}: {[round(v, 6) for v in fc[:, i]]}')

## Summary

| Step | Tool / Function |
|---|---|
| Differencing for stationarity | `timeseries.difference` |
| ACF analysis | `timeseries.autocorrelation` |
| VAR fitting & forecasting | `timeseries.VARModel` (numpy) |